# Error and forgetting analysis
Run after the core grid. This finds which validation pairs were forgotten, how concentrated the loss is, and whether length or source behavior is associated with forgetting.

In [ ]:
import os, sys, json
import pandas as pd
if os.path.exists('/content'):
    if not os.path.exists('/content/mfr-dpo'):
        !git clone -q https://github.com/prabudhd2003/mfr-dpo.git /content/mfr-dpo
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/mfr-dpo'
    DRIVE_DIR = '/content/drive/MyDrive/CSCI544/mfr-dpo'
else:
    REPO = '..'
    DRIVE_DIR = os.environ['MFR_DRIVE_DIR']
sys.path.insert(0, f'{REPO}/src')
import mfr_analysis, mfr_data
RUN_NAME = 'v2_o2_mfr_s0'
RUN_DIR = f'{DRIVE_DIR}/runs/{RUN_NAME}'
settings = json.load(open(f'{RUN_DIR}/settings.json'))
splits = mfr_data.load_splits(f'{REPO}/data/v2')


In [ ]:
order = settings['order']
for learned_at, dataset in enumerate(order[:-1], start=1):
    learned_dir = mfr_analysis.stage_folder(RUN_DIR, order, learned_at, settings.get('stage1_from'))
    learned = pd.read_csv(f'{learned_dir}/margins_{dataset}_val.csv')
    final = pd.read_csv(f'{RUN_DIR}/stage3_{order[-1]}/margins_{dataset}_val.csv')
    frame = splits[dataset]['val'][['id','prompt','prompt_tokens','chosen_tokens','rejected_tokens']].merge(
        learned[['id','margin']], on='id').merge(final[['id','margin']], on='id', suffixes=('_learned','_final'))
    frame['forgetting'] = frame.margin_learned - frame.margin_final
    frame['max_length'] = frame.prompt_tokens + frame[['chosen_tokens','rejected_tokens']].max(axis=1)
    print('\n', dataset, 'accuracy-point loss is in notebook 07')
    print('Top 10% share of positive margin loss:', round(100*mfr_analysis.concentration(frame.forgetting), 1), '%')
    display(frame.sort_values('forgetting', ascending=False).head(10))
    print(frame[['forgetting','max_length']].corr().round(3))
